In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
import numpy as np
import os

# Configuration
BATCH_SIZE = 4
IMG_HEIGHT = 128
IMG_WIDTH = 128
TRAIN_DIR = 'dataset/train'
TEST_DIR = 'dataset/test'

print("TensorFlow version:", tf.__version__)

In [ ]:
# Load training data
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names
print("Classes found:", class_names)

# Normalization layer
normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))

In [ ]:
model = models.Sequential([
    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
# Train for 5 epochs
history = model.fit(train_ds, epochs=5)

In [ ]:
def evaluate_test_folder(test_dir):
    print(f"\n{'='*10} EVALUATION {'='*10}")
    
    for class_name in os.listdir(test_dir):
        class_path = os.path.join(test_dir, class_name)
        if not os.path.isdir(class_path): continue
        
        for img_file in os.listdir(class_path):
            img_path = os.path.join(class_path, img_file)
            
            # Prepare image
            img = image.load_img(img_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
            img_array = image.img_to_array(img) / 255.0
            img_array = np.expand_dims(img_array, axis=0)
            
            # Predict
            predictions = model.predict(img_array, verbose=0)
            predicted_index = np.argmax(predictions)
            predicted_label = class_names[predicted_index]
            
            # Check results
            if predicted_label != class_name:
                print(f"❌ MISCLASSIFIED: {img_file}")
                print(f"   Actual: {class_name} | Predicted: {predicted_label}")
            else:
                print(f"✅ Correct: {img_file} ({predicted_label})")

evaluate_test_folder(TEST_DIR)